# NucSeq Age Correlation: Fisher's Z Comparison (Normal vs MASLD)

**분석 목적**: `Liver_Nucseq_age_bin_analysis.ipynb`에서 생성된 Spearman correlation CSV를 입력으로,
BH adj_pval < 0.05 & |r| > 0.2 필터링 후 Fisher's Z transformation으로
Normal(Healthy)과 MASLD 간 age correlation 계수 차이가 유의한 유전자를 선별하고 패턴을 분류·시각화합니다.

**분석 순서**
1. CSV 로드 → BH 보정
2. adj_pval < 0.05 & |r| > 0.2 필터링
3. Fisher's Z transformation 검정 (Normal vs MASLD)
4. 유의한 유전자 선별 (Fisher adj_p < 0.05)
5. 패턴 분류 및 시각화
6. 결과 Excel 저장

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import norm
from scipy.stats import t as tdist
from scipy.optimize import brentq
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
print('Imports OK')

## 1. Configuration

In [ ]:
# ─── 경로 설정 ───────────────────────────────────────────────
ANALYSIS_DIR = '/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/02_analysis'
OUTPUT_DIR   = '/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/03_script/Fisher_Z_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── 분석 파라미터 ────────────────────────────────────────────
METHOD           = 'Spearman'  # 'Spearman' or 'Pearson'
ADJ_PVAL_CUTOFF  = 0.05        # BH 보정 p-value 컷오프
R_CUTOFF         = 0.2         # |r| 최소값
FISHER_PADJ_CUT  = 0.05        # Fisher's Z BH 보정 후 컷오프

# 분석할 셀타입 (All = 전체 세포)
# key: 출력 레이블, value: 파일 suffix (None이면 celltype suffix 없음)
CELLTYPE_MAP = {
    'All'          : None,
    'B_cells'      : 'B_cells',
    'Cholangiocyte': 'Cholangiocyte',
    'Endothelial'  : 'Endothelial',
    'Hepatocyte'   : 'Hepatocyte',
    'Lymphocyte'   : 'Lymphocyte',
    'Macrophage'   : 'Macrophage',
    'Stellete'     : 'Stellete',
}

# 패턴 색상 정의
PATTERN_COLORS = {
    'Both_pos_MASLD_stronger' : '#e6474a',   # 빨강 – MASLD에서 양의 노화 상관 강화
    'Both_pos_Normal_stronger': '#f79646',   # 주황 – MASLD에서 양의 노화 상관 약화
    'Both_neg_MASLD_stronger' : '#4472c4',   # 파랑 – MASLD에서 음의 노화 상관 강화
    'Both_neg_Normal_stronger': '#70b0e0',   # 연파랑 – MASLD에서 음의 노화 상관 약화
    'Normal_pos_MASLD_neg'    : '#7030a0',   # 보라 – 방향 역전 (Normal+, MASLD-)
    'Normal_neg_MASLD_pos'    : '#00b050',   # 초록 – 방향 역전 (Normal-, MASLD+)
    'Other'                   : '#cccccc',   # 회색
}
PATTERN_ORDER = list(PATTERN_COLORS.keys())

r_col       = 'spearman_r' if METHOD == 'Spearman' else 'pearson_r'
file_prefix = 'Nucseq_age_correlation' if METHOD == 'Spearman' else 'Nucseq_age_pearson'

print(f'Method      : {METHOD}')
print(f'R column    : {r_col}')
print(f'Adj p cutoff: {ADJ_PVAL_CUTOFF}')
print(f'|r| cutoff  : {R_CUTOFF}')
print(f'Fisher padj : {FISHER_PADJ_CUT}')
print(f'Output dir  : {OUTPUT_DIR}')

## 2. 분석 함수 정의

In [ ]:
def load_and_bh_correct(filepath, r_col):
    """CSV 로드 후 BH FDR 보정하여 adj_pval 컬럼 추가."""
    df = pd.read_csv(filepath)
    df = df.dropna(subset=[r_col, 'pval']).reset_index(drop=True)
    _, adj_pvals, _, _ = multipletests(df['pval'], method='fdr_bh')
    df['adj_pval'] = adj_pvals
    return df


def estimate_n_from_data(df, r_col='spearman_r', p_col='pval', n_samples=200):
    """
    p-value와 r 값으로부터 N을 역산.
    Spearman/Pearson t-통계량: t = r * sqrt((n-2)/(1-r^2)), df=n-2
    """
    sample = df.dropna(subset=[r_col, p_col])
    sample = sample[
        (sample[r_col].abs() >= 0.003) &
        (sample[r_col].abs() <= 0.12)  &
        (sample[p_col]  > 1e-100)      &
        (sample[p_col]  < 0.3)
    ]
    if len(sample) < 5:
        return None
    sample = sample.sample(min(n_samples, len(sample)), random_state=42)

    n_estimates = []
    for _, row in sample.iterrows():
        r, p = row[r_col], row[p_col]
        if abs(r) >= 1.0:
            continue
        def obj(n, r=r, p=p):
            t = r * np.sqrt((n - 2) / (1 - r**2))
            p_calc = 2 * tdist.sf(abs(t), df=n - 2)
            return np.log(p_calc + 1e-300) - np.log(p + 1e-300)
        try:
            n_est = brentq(obj, 4, 2e7, xtol=1.0)
            if 10 < n_est < 2e7:
                n_estimates.append(n_est)
        except Exception:
            pass

    return int(np.median(n_estimates)) if n_estimates else None


def fisher_z_test(r1_arr, r2_arr, n1, n2):
    """
    Fisher's Z transformation으로 두 상관계수 차이 검정 (벡터화).
    H0: rho1 == rho2 (양측 검정)

    반환: z_stat, p_val (raw, BH 보정 전)
    """
    r1 = np.clip(np.asarray(r1_arr, dtype=float), -0.99999, 0.99999)
    r2 = np.clip(np.asarray(r2_arr, dtype=float), -0.99999, 0.99999)
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    z_stat = (z1 - z2) / se
    p_val  = 2 * norm.sf(np.abs(z_stat))
    return z_stat, p_val


def classify_pattern(r_n, r_m):
    """Normal vs MASLD r 값 방향 및 크기 패턴 분류."""
    if r_n > 0 and r_m > 0:
        return 'Both_pos_MASLD_stronger' if r_m > r_n else 'Both_pos_Normal_stronger'
    elif r_n < 0 and r_m < 0:
        return 'Both_neg_MASLD_stronger' if r_m < r_n else 'Both_neg_Normal_stronger'
    elif r_n > 0 and r_m < 0:
        return 'Normal_pos_MASLD_neg'
    elif r_n < 0 and r_m > 0:
        return 'Normal_neg_MASLD_pos'
    return 'Other'


print('함수 정의 완료')

## 3. 셀타입별 Fisher's Z 분석

In [ ]:
all_results     = {}   # ct -> {'filtered': df, 'sig': df, 'n_healthy': int, 'n_masld': int}
n_summary_rows  = []

for ct, ct_suffix in CELLTYPE_MAP.items():
    # 파일 경로 구성
    if ct_suffix is None:
        h_path = os.path.join(ANALYSIS_DIR, f'{file_prefix}_Healthy.csv')
        m_path = os.path.join(ANALYSIS_DIR, f'{file_prefix}_MASLD.csv')
    else:
        h_path = os.path.join(ANALYSIS_DIR, f'{file_prefix}_Healthy_{ct_suffix}.csv')
        m_path = os.path.join(ANALYSIS_DIR, f'{file_prefix}_MASLD_{ct_suffix}.csv')

    if not os.path.exists(h_path) or not os.path.exists(m_path):
        print(f'[SKIP] {ct}: 파일 없음')
        continue

    print(f'\n{"="*55}\n[{ct}]')

    # ── 로드 및 BH 보정 ──────────────────────────────────────
    df_h = load_and_bh_correct(h_path, r_col)
    df_m = load_and_bh_correct(m_path, r_col)
    print(f'  Healthy  : {len(df_h):>7,} 유전자')
    print(f'  MASLD    : {len(df_m):>7,} 유전자')

    # ── N 역산 ───────────────────────────────────────────────
    n_h = estimate_n_from_data(df_h, r_col=r_col)
    n_m = estimate_n_from_data(df_m, r_col=r_col)
    print(f'  N (Healthy) ≈ {n_h:,}  |  N (MASLD) ≈ {n_m:,}')
    if n_h is None or n_m is None or n_h < 4 or n_m < 4:
        print('  [SKIP] N 추정 실패')
        continue

    n_summary_rows.append({'celltype': ct, 'N_Healthy': n_h, 'N_MASLD': n_m})

    # ── 공통 유전자 병합 ─────────────────────────────────────
    merged = df_h.merge(df_m, on='gene', suffixes=('_h', '_m'))
    merged = merged.rename(columns={
        f'{r_col}_h'   : 'r_normal',
        f'{r_col}_m'   : 'r_masld',
        'pval_h'       : 'pval_normal',
        'pval_m'       : 'pval_masld',
        'adj_pval_h'   : 'adj_pval_normal',
        'adj_pval_m'   : 'adj_pval_masld',
    })

    # ── 필터링: (adj_p < 0.05 AND |r| > 0.2) in at least one condition ──
    pass_h = (merged['adj_pval_normal'] < ADJ_PVAL_CUTOFF) & (merged['r_normal'].abs() >= R_CUTOFF)
    pass_m = (merged['adj_pval_masld']  < ADJ_PVAL_CUTOFF) & (merged['r_masld'].abs()  >= R_CUTOFF)
    filtered = merged[pass_h | pass_m].copy().reset_index(drop=True)
    print(f'  필터 통과 (adj_p<{ADJ_PVAL_CUTOFF} & |r|≥{R_CUTOFF}, Normal OR MASLD): '
          f'{len(filtered):,} 유전자')

    if len(filtered) < 3:
        print('  [SKIP] 유전자 수 부족')
        continue

    # ── Fisher's Z 검정 ──────────────────────────────────────
    z_stats, p_fisher = fisher_z_test(
        filtered['r_normal'].values, filtered['r_masld'].values, n_h, n_m
    )
    filtered['fisher_z_stat']  = z_stats
    filtered['fisher_pval']    = p_fisher
    _, adj_fisher, _, _ = multipletests(p_fisher, method='fdr_bh')
    filtered['fisher_adj_pval'] = adj_fisher
    filtered['delta_r']         = filtered['r_masld'] - filtered['r_normal']

    # ── 패턴 분류 ────────────────────────────────────────────
    filtered['pattern'] = [
        classify_pattern(rn, rm)
        for rn, rm in zip(filtered['r_normal'], filtered['r_masld'])
    ]

    # ── 유의한 유전자 ─────────────────────────────────────────
    sig_mask = filtered['fisher_adj_pval'] < FISHER_PADJ_CUT
    sig_df   = filtered[sig_mask].copy()
    print(f'  Fisher Z 유의 (adj_p<{FISHER_PADJ_CUT}): {len(sig_df):,} 유전자')
    for pat in PATTERN_ORDER:
        n_pat = (sig_df['pattern'] == pat).sum()
        if n_pat > 0:
            print(f'    {pat:35s}: {n_pat:>5}')

    all_results[ct] = {
        'filtered' : filtered,
        'sig'      : sig_df,
        'n_healthy': n_h,
        'n_masld'  : n_m,
    }

print('\n\n===== 전체 분석 완료 =====')
print(f'분석된 셀타입: {list(all_results.keys())}')

## 4. N 추정 및 유전자 수 요약

In [ ]:
# N 추정 요약
df_n_summary = pd.DataFrame(n_summary_rows)
print('=== N (세포 수) 추정 요약 ===')
print(df_n_summary.to_string(index=False))

# 유전자 수 요약
summary_rows = []
for ct, res in all_results.items():
    f  = res['filtered']
    s  = res['sig']
    row = {
        'celltype'       : ct,
        'filtered_genes' : len(f),
        'fisher_sig_genes': len(s),
    }
    for pat in PATTERN_ORDER:
        row[pat] = (s['pattern'] == pat).sum()
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print('\n=== 유전자 수 요약 ===')
print(df_summary.to_string(index=False))
df_summary

## 5. 시각화

### 5-1. Scatter plot: r_Normal vs r_MASLD (셀타입별)

In [ ]:
def plot_scatter_per_celltype(all_results, pattern_colors, pattern_order, output_dir,
                               fisher_padj_cut=0.05):
    """
    각 셀타입별 r_Normal vs r_MASLD scatter plot.
    - 회색: Fisher Z 비유의 유전자
    - 컬러: Fisher Z 유의 유전자 (패턴별 색상)
    """
    n_ct   = len(all_results)
    ncols  = min(3, n_ct)
    nrows  = (n_ct + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(5.5 * ncols, 5 * nrows))
    axes = np.array(axes).flatten()

    for ax_idx, (ct, res) in enumerate(all_results.items()):
        ax  = axes[ax_idx]
        f   = res['filtered']
        sig = res['sig']

        # 비유의 유전자 (회색, 뒤에)
        non_sig = f[f['fisher_adj_pval'] >= fisher_padj_cut]
        ax.scatter(non_sig['r_normal'], non_sig['r_masld'],
                   c='#cccccc', s=10, alpha=0.4, linewidths=0,
                   label=f'Non-sig (n={len(non_sig):,})', zorder=1)

        # 유의 유전자 (패턴별 색상, 앞에)
        for pat in pattern_order:
            if pat == 'Other':
                continue
            sub = sig[sig['pattern'] == pat]
            if len(sub) == 0:
                continue
            ax.scatter(sub['r_normal'], sub['r_masld'],
                       c=pattern_colors[pat], s=18, alpha=0.8,
                       linewidths=0, label=f'{pat} (n={len(sub):,})',
                       zorder=3)

        # 기준선 & 대각선
        ax.axhline(0, color='black', linewidth=0.7, linestyle='--', zorder=2)
        ax.axvline(0, color='black', linewidth=0.7, linestyle='--', zorder=2)
        lim = max(
            f['r_normal'].abs().max(), f['r_masld'].abs().max(), 0.25
        ) * 1.1
        ax.plot([-lim, lim], [-lim, lim],
                color='grey', linewidth=0.7, linestyle=':', alpha=0.5, zorder=2)

        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel('Normal (Healthy) Spearman r', fontsize=10)
        ax.set_ylabel('MASLD Spearman r', fontsize=10)
        ax.set_title(f'{ct}\nFisher sig: {len(sig):,} / {len(f):,}',
                     fontsize=11, fontweight='bold')
        ax.legend(loc='upper left', fontsize=6.5, framealpha=0.7,
                  markerscale=1.2, ncol=1)
        ax.set_aspect('equal', adjustable='box')
        sns.despine(ax=ax)

    for ax in axes[len(all_results):]:
        ax.axis('off')

    fig.suptitle(
        f'Age Correlation: Normal vs MASLD  (adj_p<{ADJ_PVAL_CUTOFF}, |r|≥{R_CUTOFF})\n'
        f'Colored = Fisher\'s Z sig (adj_p<{fisher_padj_cut})',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    out = os.path.join(output_dir, 'scatter_r_Normal_vs_MASLD.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


plot_scatter_per_celltype(all_results, PATTERN_COLORS, PATTERN_ORDER, OUTPUT_DIR, FISHER_PADJ_CUT)

### 5-2. Stacked bar chart: 패턴별 유전자 수 (셀타입별)

In [ ]:
def plot_pattern_barplot(all_results, pattern_colors, pattern_order, output_dir):
    """
    셀타입 × 패턴별 Fisher Z 유의 유전자 수 stacked bar chart.
    """
    pat_cols = [p for p in pattern_order if p != 'Other']
    bar_data = {}
    for ct, res in all_results.items():
        sig = res['sig']
        bar_data[ct] = {pat: (sig['pattern'] == pat).sum() for pat in pat_cols}

    df_bar = pd.DataFrame(bar_data).T.fillna(0).astype(int)
    df_bar = df_bar[pat_cols]  # 열 순서 고정

    fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                              gridspec_kw={'width_ratios': [2.5, 1]})

    # ── 왼쪽: stacked bar ───────────────────────────────────
    ax = axes[0]
    bottom = np.zeros(len(df_bar))
    xticks = np.arange(len(df_bar))
    for pat in pat_cols:
        heights = df_bar[pat].values
        ax.bar(xticks, heights, bottom=bottom,
               color=pattern_colors[pat], label=pat, edgecolor='white', linewidth=0.3)
        bottom += heights

    ax.set_xticks(xticks)
    ax.set_xticklabels(df_bar.index, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Number of significant genes', fontsize=11)
    ax.set_title('Fisher\'s Z Significant Genes by Pattern & Celltype',
                 fontsize=12, fontweight='bold')
    ax.legend(title='Pattern', bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, title_fontsize=9)
    sns.despine(ax=ax)

    # ── 오른쪽: 전체 합산 pie ─────────────────────────────────
    ax2 = axes[1]
    total_by_pat = df_bar.sum()
    total_by_pat = total_by_pat[total_by_pat > 0]
    wedge_colors = [pattern_colors[p] for p in total_by_pat.index]
    ax2.pie(
        total_by_pat.values,
        labels=[f'{p}\n(n={v})' for p, v in zip(total_by_pat.index, total_by_pat.values)],
        colors=wedge_colors,
        autopct='%1.1f%%',
        pctdistance=0.75,
        textprops={'fontsize': 8},
        startangle=140,
    )
    ax2.set_title('Overall Pattern Distribution\n(all celltypes combined)',
                  fontsize=11, fontweight='bold')

    plt.tight_layout()
    out = os.path.join(output_dir, 'pattern_barplot.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


plot_pattern_barplot(all_results, PATTERN_COLORS, PATTERN_ORDER, OUTPUT_DIR)

### 5-3. Volcano plot: Δr vs −log10(Fisher adj_p)

In [ ]:
def plot_volcano_per_celltype(all_results, pattern_colors, pattern_order, output_dir,
                               fisher_padj_cut=0.05, top_n=10):
    """
    x축: delta_r (r_MASLD - r_Normal)
    y축: -log10(fisher_adj_pval)
    색상: 패턴 (Fisher Z 유의 유전자)
    상위 top_n 유전자 라벨 표시
    """
    n_ct  = len(all_results)
    ncols = min(3, n_ct)
    nrows = (n_ct + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(5.5 * ncols, 4.5 * nrows))
    axes = np.array(axes).flatten()

    sig_line = -np.log10(fisher_padj_cut)

    for ax_idx, (ct, res) in enumerate(all_results.items()):
        ax  = axes[ax_idx]
        f   = res['filtered']
        sig = res['sig']
        y   = -np.log10(f['fisher_adj_pval'].clip(lower=1e-300))

        # 비유의 (회색)
        non_sig_mask = f['fisher_adj_pval'] >= fisher_padj_cut
        ax.scatter(f.loc[non_sig_mask, 'delta_r'], y[non_sig_mask],
                   c='#cccccc', s=8, alpha=0.4, linewidths=0,
                   zorder=1)

        # 유의 (패턴별 색상)
        for pat in pattern_order:
            if pat == 'Other':
                continue
            sub_idx = sig[sig['pattern'] == pat].index
            if len(sub_idx) == 0:
                continue
            sub_f = f.loc[sub_idx]
            sub_y = -np.log10(sub_f['fisher_adj_pval'].clip(lower=1e-300))
            ax.scatter(sub_f['delta_r'], sub_y,
                       c=pattern_colors[pat], s=18, alpha=0.85,
                       linewidths=0, label=f'{pat} (n={len(sub_idx):,})',
                       zorder=3)

        # 유의 기준선
        ax.axhline(sig_line, color='grey', linewidth=0.8, linestyle='--', zorder=2)
        ax.axvline(0, color='black', linewidth=0.5, linestyle=':', zorder=2)

        # 상위 유전자 라벨 (|delta_r| 기준)
        if len(sig) > 0:
            top_genes = sig.nlargest(top_n, 'delta_r')[['gene', 'delta_r', 'fisher_adj_pval']]
            top_neg   = sig.nsmallest(top_n // 2, 'delta_r')[['gene', 'delta_r', 'fisher_adj_pval']]
            to_label  = pd.concat([top_genes, top_neg]).drop_duplicates('gene').head(top_n)
            for _, row in to_label.iterrows():
                yv = -np.log10(max(row['fisher_adj_pval'], 1e-300))
                ax.annotate(
                    row['gene'],
                    xy=(row['delta_r'], yv),
                    xytext=(3, 2), textcoords='offset points',
                    fontsize=6, color='black', alpha=0.8,
                )

        ax.set_xlabel('Δr  (r_MASLD − r_Normal)', fontsize=9)
        ax.set_ylabel('−log₁₀(Fisher adj p)', fontsize=9)
        ax.set_title(f'{ct}  (Fisher sig: {len(sig):,})',
                     fontsize=11, fontweight='bold')
        ax.legend(loc='upper right', fontsize=6, framealpha=0.7)
        sns.despine(ax=ax)

    for ax in axes[len(all_results):]:
        ax.axis('off')

    fig.suptitle(
        'Volcano: Δr vs −log₁₀(Fisher adj p)  [MASLD − Normal]',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    out = os.path.join(output_dir, 'volcano_delta_r.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


plot_volcano_per_celltype(all_results, PATTERN_COLORS, PATTERN_ORDER, OUTPUT_DIR,
                          FISHER_PADJ_CUT, top_n=8)

### 5-4. Heatmap: 셀타입 × 상위 유전자 (Δr)

In [ ]:
def plot_delta_r_heatmap(all_results, output_dir, top_per_ct=20):
    """
    각 셀타입에서 Fisher Z 유의 유전자 중 |delta_r| 상위 top_per_ct개를 모아
    gene × celltype heatmap (값: delta_r = r_MASLD - r_Normal).
    """
    # 각 셀타입의 top 유전자 수집
    gene_union = set()
    for ct, res in all_results.items():
        sig = res['sig']
        if len(sig) == 0:
            continue
        top = sig.nlargest(top_per_ct // 2, 'delta_r')['gene'].tolist()
        bot = sig.nsmallest(top_per_ct // 2, 'delta_r')['gene'].tolist()
        gene_union.update(top + bot)

    if not gene_union:
        print('시각화할 유전자 없음')
        return

    # delta_r 행렬 구성
    matrix = {}
    for ct, res in all_results.items():
        f = res['filtered'].set_index('gene')
        matrix[ct] = f['delta_r'].reindex(list(gene_union))

    hm_df = pd.DataFrame(matrix).dropna(how='all')

    # 클러스터링을 위한 정렬 (delta_r 합 기준)
    hm_df = hm_df.loc[hm_df.abs().sum(axis=1).sort_values(ascending=False).index]

    vmax = max(abs(hm_df.values[~np.isnan(hm_df.values)]).max(), 0.01)

    fig_h = max(6, len(hm_df) * 0.32)
    fig_w = max(5, len(hm_df.columns) * 1.2)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        hm_df.fillna(0),
        cmap='RdBu_r', center=0,
        vmin=-vmax, vmax=vmax,
        ax=ax,
        linewidths=0.3, linecolor='lightgray',
        cbar_kws={'label': 'Δr  (r_MASLD − r_Normal)', 'shrink': 0.6},
        xticklabels=True, yticklabels=True,
    )
    ax.set_title(
        f'Δr Heatmap (Fisher Z sig, top {top_per_ct} per celltype)\n'
        f'Red = MASLD stronger positive / Blue = MASLD stronger negative',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Celltype', fontsize=11)
    ax.set_ylabel('Gene', fontsize=11)
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=9, rotation=30)

    plt.tight_layout()
    out = os.path.join(output_dir, 'delta_r_heatmap.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')
    return hm_df


hm_df = plot_delta_r_heatmap(all_results, OUTPUT_DIR, top_per_ct=30)

### 5-5. Pattern proportion heatmap: 셀타입 × 패턴

In [ ]:
def plot_pattern_proportion_heatmap(all_results, pattern_colors, pattern_order, output_dir):
    """
    셀타입 × 패턴  –  각 셀타입에서 패턴별 유전자 비율 heatmap.
    """
    pat_cols = [p for p in pattern_order if p != 'Other']
    rows = []
    for ct, res in all_results.items():
        sig = res['sig']
        total = len(sig)
        if total == 0:
            rows.append({**{'celltype': ct}, **{p: 0.0 for p in pat_cols}})
        else:
            rows.append({**{'celltype': ct},
                         **{p: (sig['pattern'] == p).sum() / total for p in pat_cols}})
    df_prop = pd.DataFrame(rows).set_index('celltype')[pat_cols]

    fig, ax = plt.subplots(figsize=(len(pat_cols) * 1.6, len(df_prop) * 0.7 + 2))

    # 각 셀 값 (비율 0~1) + 주석
    sns.heatmap(
        df_prop,
        annot=True, fmt='.2f',
        cmap='YlOrRd', vmin=0, vmax=1,
        ax=ax, linewidths=0.5, linecolor='lightgray',
        cbar_kws={'label': 'Proportion', 'shrink': 0.7},
        annot_kws={'size': 8},
    )
    ax.set_title('Pattern Proportion per Celltype\n(among Fisher Z significant genes)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Pattern', fontsize=10)
    ax.set_ylabel('Celltype', fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

    plt.tight_layout()
    out = os.path.join(output_dir, 'pattern_proportion_heatmap.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


plot_pattern_proportion_heatmap(all_results, PATTERN_COLORS, PATTERN_ORDER, OUTPUT_DIR)

### 5-6. 패턴별 Top 유전자 dot plot (Hepatocyte 기준)

In [ ]:
def plot_top_genes_by_pattern(all_results, ct, pattern_colors, pattern_order,
                               output_dir, top_n=15):
    """
    특정 셀타입에서 각 패턴별 top 유전자를 |delta_r| 기준으로 수평 막대 그래프.
    """
    if ct not in all_results:
        print(f'{ct} 결과 없음')
        return

    sig = all_results[ct]['sig']
    if len(sig) == 0:
        print(f'{ct}: 유의 유전자 없음')
        return

    pat_cols = [p for p in pattern_order if p != 'Other']
    n_pats   = sum(1 for p in pat_cols if (sig['pattern'] == p).sum() > 0)
    ncols    = min(3, n_pats)
    nrows    = (n_pats + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(6 * ncols, max(3, top_n * 0.3) * nrows))
    axes = np.array(axes).flatten()

    plot_idx = 0
    for pat in pat_cols:
        sub = sig[sig['pattern'] == pat]
        if len(sub) == 0:
            continue
        sub = sub.nlargest(top_n, 'delta_r') if pat in ('Both_pos_MASLD_stronger', 'Normal_neg_MASLD_pos') \
              else sub.nsmallest(top_n, 'delta_r')
        sub = sub.sort_values('delta_r', ascending=True)

        ax = axes[plot_idx]
        bars = ax.barh(sub['gene'], sub['delta_r'],
                       color=pattern_colors[pat], edgecolor='white', linewidth=0.3)
        ax.axvline(0, color='black', linewidth=0.7)
        ax.set_xlabel('Δr (r_MASLD − r_Normal)', fontsize=9)
        ax.set_title(f'{pat}\n(n={len(sig[sig["pattern"]==pat]):,})',
                     fontsize=9, fontweight='bold', color=pattern_colors[pat])
        ax.tick_params(axis='y', labelsize=7)
        sns.despine(ax=ax)
        plot_idx += 1

    for ax in axes[plot_idx:]:
        ax.axis('off')

    fig.suptitle(f'[{ct}] Top Genes by Pattern  (Fisher Z sig)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe_ct = ct.replace('/', '_').replace(' ', '_')
    out = os.path.join(output_dir, f'top_genes_by_pattern_{safe_ct}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


# Hepatocyte 및 전체 셀 결과 시각화
for target_ct in ['Hepatocyte', 'All']:
    plot_top_genes_by_pattern(all_results, target_ct, PATTERN_COLORS, PATTERN_ORDER,
                               OUTPUT_DIR, top_n=15)

## 6. 결과 Excel 저장

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

# 패턴 → 색상 매핑 (openpyxl hex)
def hex_to_openpyxl(hex_color):
    return hex_color.lstrip('#').upper()

PATTERN_FILLS = {
    pat: PatternFill(start_color=hex_to_openpyxl(col),
                     end_color=hex_to_openpyxl(col),
                     fill_type='solid')
    for pat, col in PATTERN_COLORS.items() if pat != 'Other'
}

excel_out = os.path.join(OUTPUT_DIR, 'Fisher_Z_age_correlation_results.xlsx')
wb = Workbook()
wb.remove(wb.active)

output_cols = [
    'gene', 'r_normal', 'r_masld', 'delta_r',
    'adj_pval_normal', 'adj_pval_masld',
    'fisher_z_stat', 'fisher_pval', 'fisher_adj_pval',
    'pattern',
]

# ── 시트 1: 요약 ─────────────────────────────────────────────
ws_sum = wb.create_sheet(title='Summary')
ws_sum.append(['celltype', 'N_Healthy', 'N_MASLD', 'filtered_genes',
               'fisher_sig_genes'] + [p for p in PATTERN_ORDER if p != 'Other'])
for cell in ws_sum[1]:
    cell.font = Font(bold=True)

for ct, res in all_results.items():
    sig = res['sig']
    row = [ct, res['n_healthy'], res['n_masld'],
           len(res['filtered']), len(sig)]
    row += [(sig['pattern'] == p).sum() for p in PATTERN_ORDER if p != 'Other']
    ws_sum.append(row)

for col_idx in range(1, ws_sum.max_column + 1):
    ws_sum.column_dimensions[get_column_letter(col_idx)].width = 20

# ── 시트 2~: 셀타입별 유의 유전자 ────────────────────────────
for ct, res in all_results.items():
    sig = res['sig'][output_cols].copy()
    if len(sig) == 0:
        continue

    sheet_name = ct[:31]  # Excel 시트명 최대 31자
    ws = wb.create_sheet(title=sheet_name)

    # 헤더
    ws.append(output_cols)
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')

    # 데이터 + 패턴 색상
    for _, row_data in sig.iterrows():
        ws.append([row_data[c] for c in output_cols])
        cur_row = ws.max_row
        pat = row_data['pattern']
        if pat in PATTERN_FILLS:
            for cell in ws[cur_row]:
                cell.fill = PATTERN_FILLS[pat]

    # 열 너비
    for col_idx, col_name in enumerate(output_cols, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = max(len(col_name) + 2, 12)

    print(f'시트 [{sheet_name}]: {len(sig):,}행')

# ── 시트: 전체 합산 (모든 셀타입, Fisher Z 유의) ───────────────
ws_all = wb.create_sheet(title='All_celltypes_sig')
ws_all.append(['celltype'] + output_cols)
for cell in ws_all[1]:
    cell.font = Font(bold=True)

for ct, res in all_results.items():
    sig = res['sig'][output_cols].copy()
    for _, row_data in sig.iterrows():
        ws_all.append([ct] + [row_data[c] for c in output_cols])
        cur_row = ws_all.max_row
        pat = row_data['pattern']
        if pat in PATTERN_FILLS:
            for cell in ws_all[cur_row]:
                cell.fill = PATTERN_FILLS[pat]

for col_idx in range(1, ws_all.max_column + 1):
    ws_all.column_dimensions[get_column_letter(col_idx)].width = 16

wb.save(excel_out)
print(f'\nExcel 저장 완료: {excel_out}')

## 7. 최종 결과 요약

In [ ]:
print('=' * 60)
print('  Fisher\'s Z Age Correlation Comparison: 분석 완료')
print('=' * 60)
print(f'  Method       : {METHOD}')
print(f'  Filter       : adj_p < {ADJ_PVAL_CUTOFF}  &  |r| ≥ {R_CUTOFF}')
print(f'  Fisher cutoff: adj_p < {FISHER_PADJ_CUT}')
print()

total_sig = 0
for ct, res in all_results.items():
    sig = res['sig']
    total_sig += len(sig)
    print(f'  [{ct:15s}]  filtered={len(res["filtered"]):>7,}  '
          f'Fisher_sig={len(sig):>6,}  '
          f'N_Healthy={res["n_healthy"]:>8,}  N_MASLD={res["n_masld"]:>8,}')

print(f'\n  전체 Fisher Z 유의 유전자 (중복 포함): {total_sig:,}')
print()
print('  출력 파일:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'    {f}')